# Тема 1. Вариант 4
Борисюк Е. Г.
$$z=3x_1+x_2+4x_3+2x_4\to\min$$
$$2x_1+x_2+x_3\le8,\quad x_1+x_3+x_4=5,\quad x_2+x_4\ge4,\quad x_i\ge0.$$

### Канонический вид и вспомогательная задача
Добавляю запас $s_1$ в первое ограничение и вычитаю избыток $s_3$ из третьего. Во вторую и третью строки добавляю искусственные переменные $a_2,a_3$. Начальный базис: $s_1=8,a_2=5,a_3=4$. Вспомогательная цель: $w=a_2+a_3$.


In [1]:
from fractions import Fraction as F  # Точные дроби без округления

def canonicalize(A, b, senses, c, direction='min'):  # Приведение ограничений к равенствам
    A = [[F(v) for v in row] for row in A]  # Копируем матрицу и переводим числа в дроби
    b = list(map(F, b)); senses = list(senses)  # Правые части в дробях; копия знаков
    n = len(c); names = [f'x{i+1}' for i in range(n)]  # Число исходных переменных и их имена
    for i in range(len(b)):  # Перебираем ограничения
        if b[i] < 0:  # Проверяем знак правой части
            A[i] = [-v for v in A[i]]; b[i] *= -1  # Умножаем всю строку на минус один
            senses[i] = {'<=':'>=','>=':'<=','=':'='}[senses[i]]  # Меняем направление неравенства
    for i,s in enumerate(senses):  # Добавляем переменные к неравенствам
        if s not in ('<=','>=','='): raise ValueError('Unknown relation')  # Проверяем допустимость знака
        if s != '=':  # Равенству дополнительный столбец не нужен
            for j,row in enumerate(A): row.append(F((1 if s=='<=' else -1) if i==j else 0))  # Запас: +1; избыток: -1; в остальных строках 0
            names.append(f's{i+1}')  # Имя дополнительной переменной по номеру строки
    cost = [F(v)*(1 if direction=='min' else -1) for v in c] + [F(0)]*(len(names)-n)  # Цель для минимума; нули для дополнительных переменных
    return A,b,cost,names  # Возвращаем каноническую задачу

def auxiliary_problem(A,b,names):  # Строим начальный базис и вспомогательную цель
    m=len(b); n=len(names); basis=[]; missing=[]  # Размеры задачи, базис и строки без базиса
    for i in range(m):  # Ищем базис для каждой строки
        cols=[j for j,name in enumerate(names) if name.startswith('s') and  # Проверяем столбцы дополнительных переменных
              all(A[k][j]==int(k==i) for k in range(m))]  # Единица в своей строке, нули в остальных
        if cols: basis.append(cols[0])  # Берём найденный единичный столбец
        else: basis.append(n+len(missing)); missing.append(i)  # Иначе назначаем искусственную переменную
    T=[row+[F(int(i==k)) for k in missing]+[b[i]] for i,row in enumerate(A)]  # Добавляем искусственные столбцы и правые части
    return T,basis,[F(0)]*n+[F(1)]*len(missing),names+[f'a{k+1}' for k in missing]  # Цель: сумма искусственных переменных



### Симплекс-шаг
Считаю оценки $r_j=c_j-c_B^T T_j$. При отрицательной оценке выбираю входящий столбец, затем строку по минимальному отношению $b_i/t_{ik}$ при $t_{ik}>0$. Пересчитываю таблицу и сохраняю шаг.

In [2]:
def reduced_costs(T,basis,cost):  # Оценки и текущее значение цели
    r=[cost[j]-sum(cost[basis[i]]*T[i][j] for i in range(len(T))) for j in range(len(cost))]  # Оценка: коэффициент цели минус вклад базиса
    z=sum(cost[basis[i]]*T[i][-1] for i in range(len(T)))  # Цель: сумма коэффициентов базиса на правые части
    return r,z  # Возвращаем оценки и цель

def pivot(T,basis,row,col):  # Пересчёт по разрешающему элементу
    p=T[row][col]; T[row]=[v/p for v in T[row]]  # Нормируем разрешающую строку
    for i in range(len(T)):  # Перебираем строки таблицы
        if i!=row:  # Разрешающую строку уже обработали
            k=T[i][col]; T[i]=[v-k*w for v,w in zip(T[i],T[row])]  # Обнуляем входящий столбец в остальных строках
    basis[row]=col  # Меняем базисную переменную

def simplex(T,basis,cost,names,phase,history):  # Повторяем шаги до оптимальности
    for step in range(1000):  # Защита от слишком долгого выполнения
        r,z=reduced_costs(T,basis,cost)  # Считаем оценки текущей таблицы
        entry={'phase':phase,'step':step,'names':names[:],'basis':[names[i] for i in basis],  # Сохраняем фазу, шаг и имена базиса
               'T':[[str(v) for v in row] for row in T],'r':[str(v) for v in r],'z':str(z)}  # Снимок таблицы, оценок и цели
        history.append(entry)  # Добавляем таблицу в историю
        cols=[j for j,v in enumerate(r) if v<0]  # Столбцы, позволяющие уменьшить цель
        if not cols: return  # Нет отрицательных оценок: оптимум
        col=min(cols)  # Первый подходящий столбец по индексу
        rows=[i for i in range(len(T)) if T[i][col]>0]  # Берём только положительные элементы столбца
        if not rows: raise ValueError('Objective unbounded')  # Нет ограничивающей строки: цель не ограничена
        row=min(rows,key=lambda i:(T[i][-1]/T[i][col],basis[i]))  # Минимальное отношение; при равенстве индекс базиса
        entry['enter']=names[col]; entry['leave']=names[basis[row]]  # Запоминаем входящую и выходящую переменные
        entry['ratios']=[str(T[i][-1]/T[i][col]) if T[i][col]>0 else '—' for i in range(len(T))]  # Сохраняем отношения для допустимых строк
        pivot(T,basis,row,col)  # Выполняем разрешающий шаг
    raise RuntimeError('Iteration limit')  # Сообщаем о превышении числа шагов



### Переход к основной задаче
Если минимум вспомогательной цели равен нулю, удаляю искусственные переменные и возвращаю исходную цель. Когда все оценки неотрицательны, считываю оптимальный план из базиса.

In [3]:
def main_problem(T,basis,n):  # Подготовка таблицы к основной задаче

    for i in range(len(T)-1,-1,-1):  # Идём с конца, чтобы безопасно удалять строки
        if basis[i]>=n:  # Искусственная переменная ещё в базисе?
            if T[i][-1]!=0: raise ValueError('Infeasible')  # Ненулевая искусственная переменная: несовместность
            cols=[j for j in range(n) if j not in basis and T[i][j]!=0]  # Ищем обычную переменную для замены
            if cols: pivot(T,basis,i,min(cols))  # Выводим искусственную переменную из базиса
            else: T.pop(i); basis.pop(i)  # Удаляем избыточную строку 0 = 0
    return [row[:n]+[row[-1]] for row in T],basis  # Удаляем искусственные столбцы, сохраняем b

def solve_lp(A,b,senses,c,direction='min'):  # Объединяем все этапы решения
    C,rhs,cost,names=canonicalize(A,b,senses,c,direction)  # Получаем канонический вид
    T,basis,aux_cost,aux_names=auxiliary_problem(C,rhs,names)  # Строим вспомогательную задачу
    history=[]; simplex(T,basis,aux_cost,aux_names,'I',history)  # Запускаем первую фазу и сохраняем историю
    if reduced_costs(T,basis,aux_cost)[1]!=0: raise ValueError('Infeasible')  # Вспомогательная цель должна стать нулём
    T,basis=main_problem(T,basis,len(names))  # Удаляем искусственные переменные
    simplex(T,basis,cost,names,'II',history)  # Решаем исходную задачу во второй фазе
    x=[F(0)]*len(names)  # Небазисные переменные равны нулю
    for i,j in enumerate(basis): x[j]=T[i][-1]  # Базисные значения берём из правого столбца
    return x[:len(c)],sum(F(v)*u for v,u in zip(c,x)),history  # Возвращаем исходные x, исходную цель и таблицы



### Вариант 4
Задаю коэффициенты исходной задачи. Обе фазы, промежуточные таблицы и ответ выводятся ниже.

In [4]:
A = [[2, 1, 1, 0], [1, 0, 1, 1], [0, 1, 0, 1]]  # Коэффициенты трёх ограничений варианта 4
b = [8, 5, 4]  # Правые части ограничений
senses = ['<=', '=', '>=']  # Знаки: неравенство, равенство, неравенство
c = [3, 1, 4, 2]  # Коэффициенты исходной цели
x, z, history = solve_lp(A, b, senses, c)  # Запускаем собственный симплекс-метод

for t in history:  # Печатаем сохранённые таблицы
    print('Фаза', t['phase'], 'шаг', t['step'])  # Номер фазы и шага
    print('Базис | ' + ' | '.join(t['names']) + ' | b')  # Заголовки столбцов
    for name, row in zip(t['basis'], t['T']):  # Базисная переменная и её строка
        print(name + ' | ' + ' | '.join(row))  # Печатаем коэффициенты строки
    print('r | ' + ' | '.join(t['r']), 'Цель:', t['z'])  # Печатаем оценки и цель
    if 'enter' in t:  # Был ли переход из этой таблицы?
        print('Входит:', t['enter'], 'Выходит:', t['leave'])  # Показываем смену базиса
    print()  # Пустая строка между таблицами
print('x* =', tuple(map(str, x)))  # Оптимальная точка
print('z =', z)  # Оптимальное значение цели


Фаза I шаг 0
Базис | x1 | x2 | x3 | x4 | s1 | s3 | a2 | a3 | b
s1 | 2 | 1 | 1 | 0 | 1 | 0 | 0 | 0 | 8
a2 | 1 | 0 | 1 | 1 | 0 | 0 | 1 | 0 | 5
a3 | 0 | 1 | 0 | 1 | 0 | -1 | 0 | 1 | 4
r | -1 | -1 | -1 | -2 | 0 | 1 | 0 | 0 Цель: 9
Входит: x1 Выходит: s1

Фаза I шаг 1
Базис | x1 | x2 | x3 | x4 | s1 | s3 | a2 | a3 | b
x1 | 1 | 1/2 | 1/2 | 0 | 1/2 | 0 | 0 | 0 | 4
a2 | 0 | -1/2 | 1/2 | 1 | -1/2 | 0 | 1 | 0 | 1
a3 | 0 | 1 | 0 | 1 | 0 | -1 | 0 | 1 | 4
r | 0 | -1/2 | -1/2 | -2 | 1/2 | 1 | 0 | 0 Цель: 5
Входит: x2 Выходит: a3

Фаза I шаг 2
Базис | x1 | x2 | x3 | x4 | s1 | s3 | a2 | a3 | b
x1 | 1 | 0 | 1/2 | -1/2 | 1/2 | 1/2 | 0 | -1/2 | 2
a2 | 0 | 0 | 1/2 | 3/2 | -1/2 | -1/2 | 1 | 1/2 | 3
x2 | 0 | 1 | 0 | 1 | 0 | -1 | 0 | 1 | 4
r | 0 | 0 | -1/2 | -3/2 | 1/2 | 1/2 | 0 | 1/2 Цель: 3
Входит: x3 Выходит: x1

Фаза I шаг 3
Базис | x1 | x2 | x3 | x4 | s1 | s3 | a2 | a3 | b
x3 | 2 | 0 | 1 | -1 | 1 | 1 | 0 | -1 | 4
a2 | -1 | 0 | 0 | 2 | -1 | -1 | 1 | 1 | 1
x2 | 0 | 1 | 0 | 1 | 0 | -1 | 0 | 1 | 4
r | 1 | 0 